### Imports

In [40]:
import os
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

### Read files

In [41]:
X_train = pd.read_csv('X_train.csv',index_col='ROW_ID')
y_train = pd.read_csv('y_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('X_test.csv',index_col='ROW_ID')

### Initial exploration

In [42]:
print(X_train.head(3)) # print the data structure to double check, some NaN are present

               TS     ALLOCATION    RET_20    RET_19    RET_18    RET_17  \
ROW_ID                                                                     
0       DATE_0001  ALLOCATION_01 -0.018192 -0.000306 -0.006881 -0.002393   
1       DATE_0001  ALLOCATION_02 -0.006394 -0.001059  0.001565  0.000033   
2       DATE_0001  ALLOCATION_03 -0.016587 -0.004517 -0.005306  0.004314   

          RET_16    RET_15    RET_14    RET_13  ...  SIGNED_VOLUME_8  \
ROW_ID                                          ...                    
0       0.000507 -0.001270 -0.002539  0.002830  ...         0.818730   
1       0.002829  0.001725  0.000875 -0.002160  ...        -1.390336   
2       0.006471 -0.005868 -0.005030 -0.001488  ...         0.961318   

        SIGNED_VOLUME_7  SIGNED_VOLUME_6  SIGNED_VOLUME_5  SIGNED_VOLUME_4  \
ROW_ID                                                                       
0              0.941014         0.714129        -0.323847         0.525097   
1             -0.651784 

In [43]:
print(X_test.head(3)) # same situations

               TS     ALLOCATION    RET_20    RET_19    RET_18    RET_17  \
ROW_ID                                                                     
527073  DATE_2523  ALLOCATION_01 -0.000523  0.000066 -0.001601  0.002030   
527074  DATE_2523  ALLOCATION_02  0.002868 -0.000870  0.000609  0.003156   
527075  DATE_2523  ALLOCATION_03 -0.000435  0.003141  0.001410 -0.011531   

          RET_16    RET_15    RET_14    RET_13  ...  SIGNED_VOLUME_8  \
ROW_ID                                          ...                    
527073 -0.000137 -0.001506  0.000479 -0.001006  ...        -1.124653   
527074  0.001140  0.000728  0.002079  0.000126  ...        -1.310546   
527075 -0.000685  0.002088 -0.003898  0.008467  ...         1.402281   

        SIGNED_VOLUME_7  SIGNED_VOLUME_6  SIGNED_VOLUME_5  SIGNED_VOLUME_4  \
ROW_ID                                                                       
527073        -1.263611        -1.177919        -1.251840        -1.242161   
527074        -0.925154 

In [44]:
print(y_train.head(3)) # fast print check on target data structure

          target
ROW_ID          
0       0.009210
1       0.000405
2       0.005967


In [45]:
print(X_train.shape, X_test.shape) # look at the shape of the data
print("portfolios:", X_train["ALLOCATION"].nunique(), X_test["ALLOCATION"].nunique()) # 
# we have the same number of portfolio 278 and same number of columns in each database, good.

(527073, 44) (31870, 44)
portfolios: 278 278


In [46]:
X_train["target"] = y_train["target"]      # we attach the next-day return to each row
print("share of up days:", (X_train["target"] > 0).mean().round(4)) # we see how often we have up and down days

share of up days: 0.5072


In [47]:
RET = [f"RET_{i}" for i in range(1, 21)]   # RET_1 = most recent day
# look at the number of Nan, i.e., missing days. We will have to deal with them
print("missing RET values: train", X_train[RET].isna().sum().sum(), " test", X_test[RET].isna().sum().sum()) 

missing RET values: train 262  test 294


### Data cleaning & sorting

Here, the idea is relatively simple but it is quite important that we execute it properly. 

Primo. We are looking at returns in 20 days window and trying to predict the sign (i.e., up or down) of the next day. For each portfolio, of course.

Secondo, we note that for each portfolio tomorrow's row in our database (if present) should hold today's return shifted by one slot. To wit, today's RET_1 $\sim$ RET_2 of tomorrow, today's RET_2 $\sim$ RET_3 of tomorrow, and so on. The match is assumed to be approximate, not exact, due to the drift in the portfolio allocation through each day. 

Therefore, we have a non-trivial issue. We would like to work with a **honest train/validation split** but this would be quite hard as it is right now. Indeed, a row's target (tomorrow's return) appears as RET_1 in the next date's row. Thus, if we were to split dates at random then every validation date would have its neighbours in the training set. The trained model would effectively "see" part of the answer. *This is not allowed, of course. It would inflate  the validation score of the model with no real merit.*

Also, it is not a bad idea to possibly have time-varying parameters given the non-stationarity of the markets. So there is that too.

We thus try to build longer windows of data by looking at rows whose shifted windows matches another one. We expect to find some, but not all.

In [48]:
# We begin by looking at a single portfolio
overlap = 15 # number of days we compare (to be checked/tuned later)
g = X_train[X_train["ALLOCATION"] == "ALLOCATION_01"]
R = np.nan_to_num(g[RET].to_numpy()) # the rows, i.e., the dates of the portfolio

today = R[:,0:overlap]        # RET 1 - 15
tomorrow = R[:,1:overlap+1]   # RET 2 - 16

# Since we expect drifts, a good practice is to normalise our windows
epsilon = 1e-12  # to avoid divergences
today = today/(np.linalg.norm(today, axis =1, keepdims = True) + epsilon)
tomorrow = tomorrow/(np.linalg.norm(tomorrow, axis =1, keepdims = True) + epsilon)

# we then extract the 2 rows whose shifted window is closest

dist, idx = cKDTree(tomorrow).query(today, k =2)

# if we have a clear winner
ratio = 0.3     # to be checked/tuned later
clear = dist[:,0] < ratio*dist[:,1]
print("rows:", len(g), "| rows with a clear next-day match:", clear.sum())


# let us look at a few matches to make sure 
m = pd.DataFrame({
    "date": g["TS"].values,
    "next_date": g["TS"].values[idx[:, 0]],
    "RET_1 today": g["RET_1"].values,
    "RET_2 of match": g["RET_2"].values[idx[:, 0]],
    "clear": clear,
})
print(m[m["clear"]].head(10))

rows: 1431 | rows with a clear next-day match: 1131
         date  next_date  RET_1 today  RET_2 of match  clear
0   DATE_0001  DATE_0404     0.003944        0.003131   True
2   DATE_0005  DATE_2141     0.005628        0.006723   True
3   DATE_0007  DATE_1332    -0.003151       -0.005836   True
4   DATE_0010  DATE_0343     0.005724        0.005943   True
5   DATE_0014  DATE_2494    -0.002946       -0.003865   True
6   DATE_0015  DATE_2501    -0.001909       -0.002236   True
8   DATE_0017  DATE_1488    -0.000375        0.001183   True
9   DATE_0019  DATE_2048    -0.001959       -0.003233   True
10  DATE_0023  DATE_1617     0.001064        0.000818   True
11  DATE_0026  DATE_1312     0.000855        0.001245   True


In [ ]:
# We run for one portfolio to check it made sense, now 
# we build a function.

def next_day_votes(X, k=1, overlap = 15, ratio = 0.3):
    # For every portfolio, each row votes for the date whose row looks like shifted by k days.
    # Returns one vote per each clear match (TS --> next TS)
    votes = []
    for a, g in X.groupby("ALLOCATION"):
        R = np.nan_to_num(g[RET].to_numpy()) 
        original = R[:,0:overlap]/(np.linalg.norm(R[:,0:overlap])+epsilon)      
        shifted  = R[:,k:overlap+k]/(np.linalg.norm(R[:,k:overlap+k])+epsilon)      
        d, j = cKDTree(shifted).query(original, k =2)
        clear = d[:,0] < ratio*d[:,1]
        votes.append(pd.DataFrame({"ALLOCATION": a, "TS":g["TS"].values[clear],"NEXT_TS": g["TS"].values[j[clear,0]]}))
    return pd.concat(votes, ignore_index = True)

votes = next_day_votes(X_train, k=1, overlap=overlap, ratio= ratio)

# let us look at what we got, clear number of votes, how many recieved a vote etc. etc.
print("rows:", len(X_train), "| clear votes:", len(votes), f"({len(votes) / len(X_train):.1%})")
print("dates that received at least one vote:", votes["TS"].nunique(), "of", X_train["TS"].nunique())
print(votes.head())

rows: 527073 | clear votes: 391435 (74.3%)
dates that received at least one vote: 2519 of 2522
      ALLOCATION         TS    NEXT_TS
0  ALLOCATION_01  DATE_0001  DATE_0404
1  ALLOCATION_01  DATE_0005  DATE_2141
2  ALLOCATION_01  DATE_0007  DATE_1332
3  ALLOCATION_01  DATE_0010  DATE_0343
4  ALLOCATION_01  DATE_0014  DATE_2494


In [ ]:
# Now we have to turn the votes into links between dates

def pick_links(votes, min_votes = 10):
    # we keep a match only if enough portfolios agree (i.e., the majority), and no
    # other date claims the same next date
    n = votes.groupby(["TS","NEXT_TS"]).size().rename("n").reset_index() # introduce n column essentially
    n["share"] = n["n"]/n.groupby("TS")["n"].transform("sum")   # extract share
    best = n.sort_values("n", ascending=False).drop_duplicates("TS")       # top candidate per date
    best = best[(best["n"]>min_votes) & (best["share"]>0.5)]
    best.drop_duplicates("NEXT_TS", keep = False)
    return dict(zip(best["TS"], best["NEXT_TS"])), n 

links, vote_table = pick_links(votes)

print("dates with a kept link:", len(links), "of", X_train["TS"].nunique())

# all good but are the majorities actually fragile? 
top = vote_table.sort_values("n", ascending=False).drop_duplicates("TS")
print("share of votes for the winner, quantiles:", top["share"].quantile([0.05, 0.25, 0.5]).round(2).to_dict())


dates with a kept link: 2499 of 2522
share of votes for the winner, quantiles: {0.05: 0.75, 0.25: 1.0, 0.5: 1.0}


We see that even the weakest 5% of winners took 75% of their date's votes, so most links are clear-cut. The aggregation and vote by majority seems robust.

But we have no trust nonetheless and we want to check our links. We remark that if date $i$ follows date $j$, then each portfolio's return on $j$ should show up as its RET_2 on $i$. These should line up between $i$ and $j$ across all the portfolios the two dates share.

How do we test this? Like this: (i) we compute the Spearman (rank) correlation between the shared portfolio and multiply by $\sqrt{n}$; (ii) we keep a link if it reaches a $5\sigma$ signficance (particle physics dictates here) with respect to the $\mathcal{N}(0,1)$ expecation of a random null. Also, note that here we employ the Spearman to avoid the rare fat-tailed days to fully determine the results. Fat-tails are crucial in delaing with markets but here we want to look at a bulk properties so we weight them down. Also, we only consider cases in which at least 10 portfolios are available. Below is counting crisps.

In [57]:
def verify_links(links, k, X, min_z =5.0):
    # we create a small table per date, indexed by portoflio
    by_date = {t: g.set_index("ALLOCATION") for t, g in X.groupby("TS")}
    good, zs = {}, {}
    for t, u in links.items():
        a = by_date[t]["RET_1"]                         # day t's return
        b = by_date[u][f"RET_{k+1}"].reindex(a.index)   # same day but seen from the u date
        m = a.notna() & b.notna() # deal with nan
        n = m.sum()
        if n>= 10:
            z = a[m].corr(b[m], method = "spearman")*np.sqrt(n)
            zs[t] = z
            if z >= min_z:
                good[t] = u 
    print(f"shift {k}: {len(links)} candidate links, {len(good)} pass (z >= {min_z})")
    return good, pd.Series(zs)

links_ok, z = verify_links(links, k=1, X=X_train)
print(z.describe())
print("weakest links:\n", z.nsmallest(5))

shift 1: 2499 candidate links, 2485 pass (z >= 5.0)
count    2499.000000
mean       12.836398
std         3.305644
min         1.652326
25%        10.945831
50%        13.778941
75%        15.691396
max        16.471573
dtype: float64
weakest links:
 DATE_0474    1.652326
DATE_0500    1.837198
DATE_0449    1.856362
DATE_2423    3.407836
DATE_1811    4.133585
dtype: float64


Our test has identified a handful of weak links. However, the majority are quite strong with (median $z \approx 13.8$) and very little below $z = 5$ overall. We can therefore reject the small minority and keep the large majority quite safely.

We can now build the stratches of time of consecutive dates.

In [60]:
def build_chains(nxt,step):
    chains = []
    starts = set(nxt) - set(nxt.values()) # dates with to precedent day
    for s in starts:
        c, pos = [s], [0]
        while c[-1] in nxt:
            assert len(c) <= X_train["TS"].nunique(), f"loop detected from {s}" # check if we failed in the link constructions 
            pos.append(pos[-1] - step[c[-1]])
            c.append(nxt[c[-1]])
        chains.append((c, pos))
    chains.sort(key=lambda cp: -len(cp[0]))   # longest first
    return chains

nxt  = dict(links_ok) # only the approved links
step = {t: 1 for t in nxt}
chains = build_chains(nxt, step)

lengths = [len(c) for c, _ in chains]
placed  = {d for c, _ in chains for d in c}
print("stretches:", len(chains))
print("longest 10:", lengths[:10])
print("dates placed:", len(placed), "of", X_train["TS"].nunique())
print("dates in links but not placed (would mean a loop):", len(set(nxt) - placed))

stretches: 51
longest 10: [245, 222, 184, 183, 177, 168, 166, 162, 141, 130]
dates placed: 2518 of 2522
dates in links but not placed (would mean a loop): 0


We have (i) no loops (badl links creeping in), (ii) very good placements. So we can keep going onto **bridging**.

Essentially, a stretch of consecutive days (as we built them from above) can end either because the next day is not in our data or because a link failed the $5\sigma$ check. And yet, it is fully possible that the end oa stretch and the start of another are just $k$ days apart. In case, that is still important information of course. 

How do we go about this? 

- We only look at the links from the end to the start of stretches (reduces the search);
- We lower to overlap = 10 so that we can probe confortably up to high shifts;
- Impose an aboslute cut on the distance between the stretches, i.e, cosine similarity of at least $97\%$;
-  We raise the minimum number of votes up (bigger claim);
- Still we impose the $5\sigma$ bit;
- We forbid loops;
- We re-add the single dates as one-day stretches to see if we can still link the up.

In [ ]:
def all_chains(nxt, step, all_dates): # build all of the chains
    chains = build_chains(nxt,step)
    placed = {d for c, _ in chains for d in c} # dates which have been placed
    chains += [([d], [0]) for d in all_dates if d not in placed] # lone dates
    return chains

def unit(M): # to normalise
    return M/(np.linalg.norm(M, axis = 1 , keepdims= True) + epsilon)

def bridge_votes(X,k,from_dates,to_dates,overlap=10,max_dist =0.25): 
    # keep only candidate "from" dates and group the possible "to" dates by portfolio.
    From = X[X["TS"].isin(from_dates)]
    To = dict(tuple(X[X["TS"].isin(to_dates)].groupby("ALLOCATION")))
    votes =[]
    for a, gf in From.groupby("ALLOCATION"): 
        if a not in To:  # only compare portfolios that exist on both sides.
            continue
        gt = To[a]
        this = unit(np.nan_to_num(gf[RET].to_numpy()[:,0:overlap]))
        shifted = unit(np.nan_to_num(gf[RET].to_numpy()[:,k:overlap+k]))
        d, j = cKDTree(shifted).query(this, k=1)
        # keep only sufficiently close matches as valid bridge votes.
        ok = d < max_dist
        # record the proposed date-to-date links for the matching windows.
        votes.append(pd.DataFrame({"TS": gf["TS"].values[ok], "NEXT_TS":gt["TS"].values[j[ok]]}))
    return pd.concat(votes, ignore_index=True)

In [66]:
all_dates = X_train["TS"].unique()
chains = all_chains(nxt, step, all_dates)

In [67]:
# We cycle over increasingly long shifts to bridge gaps of k days.
for k in range(2,6):
    # identify the current end and start date of each chain
    end_of = {c[-1]: i for i, (c,_) in enumerate(chains)}
    start_of = {c[0]: i for i, (c,_) in enumerate(chains)}
    # generate portfolio-level votes for possible k-day bridges.
    v = bridge_votes(X_train, k, set(end_of), set(start_of))
    v = v[v["TS"].map(end_of) != v["NEXT_TS"].map(start_of)]  # no loop allowed   
    cand, _ = pick_links(v, min_votes=30) # get the candidates
    new, zk = verify_links(cand, k, X_train, min_z=5.0) # verify those links at 5sigma
    for t, u in new.items():
        nxt[t] = u
        step[t] = k
    # reconstruct the chains after adding the new bridges.
    chains = all_chains(nxt, step, all_dates) # extract the chains 
    print(f"  after k={k}: {len(new)} bridges, {len(chains)} stretches")


lengths = sorted((len(c) for c, _ in chains), reverse=True)
print("longest 10:", lengths[:10])
print("stretches with >= 20 dates:", sum(l >= 20 for l in lengths))

shift 2: 1 candidate links, 0 pass (z >= 5.0)
  after k=2: 0 bridges, 55 stretches
shift 3: 0 candidate links, 0 pass (z >= 5.0)
  after k=3: 0 bridges, 55 stretches
shift 4: 0 candidate links, 0 pass (z >= 5.0)
  after k=4: 0 bridges, 55 stretches
shift 5: 0 candidate links, 0 pass (z >= 5.0)
  after k=5: 0 bridges, 55 stretches
longest 10: [245, 222, 184, 183, 177, 168, 166, 162, 141, 130]
stretches with >= 20 dates: 42
